# Drone Chase Tag: davranış kontrolü (ıska analizi + videolar, GPU)

Bir self-play koşusunun iki ÖĞRENEN ajanını (varsayılan: son kovalayan vs son kaçan) karşılaştırır:
- **Iska analizi** (`scripts/near_miss.py`): kovalayan kaçana 0,5 m'den fazla yaklaşıp yakalayamadığında ne oluyor?
  Kaçan sınıra (zemin/tavan) mı yapışıyor, son anda mı yön değiştiriyor? Olasılıklı ve gürültüsüz.
- **Videolar** (`scripts/watch_selfplay.py`): 4 olasılıklı + 2 gürültüsüz maç; tuhaf/sömürü davranışı var mı?

1. **Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU** (yoksa L4) seç.
2. Hücreleri sırayla çalıştır. Sonuçlar Drive'a (`results/<koşu>/watch/`) yazılır ve tek zip olarak iner.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir("/content")                        # hücre nerede çalışırsa çalışsın, önce sabit bir yere geç
REPO = "https://github.com/HasaNSaygili/DroneChaseTag.git"
if os.path.isdir("/content/proje/.git"):    # daha önce indirildiyse silmeden güncelle
    !git -C /content/proje fetch -q origin && git -C /content/proje reset -q --hard origin/main
else:
    !git clone -q {REPO} /content/proje
assert os.path.isfile("/content/proje/scripts/near_miss.py"), "Kod indirilemedi: bu hücrenin çıktısını sohbete yapıştır"
os.chdir("/content/proje")
!git log --oneline -1
!pip install -q "crazyflow[gpu]==0.4.0" -e .

In [ ]:
# Bütün koşular burada kayıtlı: incelenecek self-play koşusu (experiments/ altındaki klasör adı).
# RUN'a çalıştırmak istediğinin adını yaz. CHASER/EVADER: sürüm numarası (-1 = son sürüm, 0 = başlangıç politikası).
RUNS = ["sp-empty-anchor-s0", "sp-faz1-anchor2-s0", "sp-empty-anchor2-floor-s0", "sp-faz1-anchor2-floor-s0"]
RUN = "sp-faz1-anchor2-s0"
RUN = RUN.removeprefix("watch-")  # runs/ altındaki notebook'un adı "watch-<koşu>"
CHASER, EVADER = -1, -1
GAMES_STOCH, GAMES_DET = 4, 2
OUT = "/content/drive/MyDrive/DroneChaseTag/results"
WOUT = f"/content/proje/experiments/{RUN}/watch"
os.makedirs(WOUT, exist_ok=True)
assert os.path.isdir(f"/content/proje/experiments/{RUN}"), f"experiments/{RUN} repoda yok"
print(RUN, "| kovalayan", CHASER, "vs kaçan", EVADER)

In [ ]:
# Iska analizi (olasılıklı + gürültüsüz); çıktı ekrana ve watch/near_miss.txt dosyasına
import os
os.chdir("/content/proje")
!python -W ignore scripts/near_miss.py --run experiments/{RUN} --chaser {CHASER} --evader {EVADER} --worlds 2048 2>&1 | grep -v -E "warp|spec|fromstring" | tee {WOUT}/near_miss.txt

In [ ]:
# Videolar (MuJoCo, EGL ile ekransız çizim)
import os
os.chdir("/content/proje")
!MUJOCO_GL=egl python -W ignore scripts/watch_selfplay.py --run experiments/{RUN} --chaser {CHASER} --evader {EVADER} --games {GAMES_STOCH} --out {WOUT} 2>&1 | grep -v -E "warp|spec|fromstring"
!MUJOCO_GL=egl python -W ignore scripts/watch_selfplay.py --run experiments/{RUN} --chaser {CHASER} --evader {EVADER} --games {GAMES_DET} --deterministic --out {WOUT} 2>&1 | grep -v -E "warp|spec|fromstring"

In [ ]:
# Drive'a kopyala ve tek zip olarak indir (videolar + ıska analizi)
import os, shutil, zipfile
from google.colab import files
shutil.copytree(WOUT, f"{OUT}/{RUN}/watch", dirs_exist_ok=True)
zpath = f"/content/watch-{RUN}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for name in sorted(os.listdir(WOUT)):
        z.write(f"{WOUT}/{name}", f"{RUN}/watch/{name}")
print(zpath, sorted(os.listdir(WOUT)))
files.download(zpath)